# Apartment for Rent

[Master Tree Ensemble Models course](https://www.trainindata.com/p/master-tree-ensemble-models)

In this notebook, we prepare the Apartment for Rent dataset for the course.

The data contains classified ads for apartments for rent in the USA. The goal is to predict the rent price.

Here, we load the data from the UCI Machine Learning Repository, sort it by date, keep the useful variables, set the correct data types, and turn the pets and amenities information into binary variables. Then, we save the data as a parquet file.

Dataset: [Apartment for Rent Classified](https://archive.ics.uci.edu/dataset/555/apartment+for+rent+classified)

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

# Apartment for Rent Classified dataset from the UCI Machine Learning Repository:
# https://archive.ics.uci.edu/dataset/555/apartment+for+rent+classified
url = "https://archive.ics.uci.edu/static/public/555/data.csv"
df = pd.read_csv(url, low_memory=False).drop(columns="id")
df.head()

,category,title,body,amenities,bathrooms,bedrooms,currency,fee,has_photo,pets_allowed,...,price_display,price_type,square_feet,address,cityname,state,latitude,longitude,source,time
0,housing/rent/apartment,One BR 507 & 509 Esplanade,"This unit is located at 507 & 509 Esplanade, R...",NaN,1,1,USD,No,Thumbnail,Cats,...,2195,Monthly,542,507 509 Esplanade,Redondo Beach,CA,33.8520,-118.3759,RentLingo,1.577360e+09
1,housing/rent/apartment,Three BR 146 Lochview Drive,"This unit is located at 146 Lochview Drive, Ne...",NaN,1.5,3,USD,No,Thumbnail,"Cats,Dogs",...,1250,Monthly,1500,146 Lochview Dr,Newport News,VA,37.0867,-76.4941,RentLingo,1.577360e+09
2,housing/rent/apartment,Three BR 3101 Morningside Drive,This unit is located at 3101 Morningside Drive...,NaN,2,3,USD,No,Thumbnail,NaN,...,1395,Monthly,1650,3101 Morningside Dr,Raleigh,NC,35.8230,-78.6438,RentLingo,1.577360e+09
3,housing/rent/apartment,Two BR 209 Aegean Way,"This unit is located at 209 Aegean Way, Vacavi...",NaN,1,2,USD,No,Thumbnail,"Cats,Dogs",...,1600,Monthly,820,209 Aegean Way,Vacaville,CA,38.3622,-121.9712,RentLingo,1.577360e+09
4,housing/rent/apartment,One BR 4805 Marquette NE,"This unit is located at 4805 Marquette NE, Alb...",NaN,1,1,USD,No,Thumbnail,"Cats,Dogs",...,975,Monthly,624,4805 Marquette NE,Albuquerque,NM,35.1038,-106.6110,RentLingo,1.577360e+09


In [2]:
df["time"] = pd.to_datetime(df["time"], unit="s")
df = df.set_index("time").sort_index()

# Remove useless columns
df = df.drop(
    columns=[
        "price_type",
        "price_display",
        "currency",
        "title",
        "body",
        "address",
    ]
)

# Drop rows with missing price values
df = df.dropna(subset=['price'])
df.shape

(99821, 14)

In [3]:
df.isnull().sum()

category            0
amenities       16074
bathrooms          63
bedrooms          124
fee                 0
has_photo           0
pets_allowed    60631
price               0
square_feet         0
cityname          302
state             302
latitude           25
longitude          27
source              2
dtype: int64

In [4]:
df.dtypes

category            str
amenities           str
bathrooms           str
bedrooms            str
fee                 str
has_photo           str
pets_allowed        str
price           float64
square_feet         str
cityname            str
state               str
latitude        float64
longitude       float64
source              str
dtype: object

In [5]:
mask = pd.to_numeric(df["square_feet"], errors="coerce").isna() & df["square_feet"].notna()
df.loc[mask, "square_feet"].unique()

<ArrowStringArray>
['NJ', 'CA']
Length: 2, dtype: str

In [6]:
df["square_feet"] = pd.to_numeric(df["square_feet"], errors="coerce")
df["bedrooms"] = pd.to_numeric(df["bedrooms"], errors="coerce").astype("Int64")
df["bathrooms"] = pd.to_numeric(df["bathrooms"], errors="coerce")
categorical_cols = [
    "category",
    "fee",
    "has_photo",
    "cityname",
    "state",
    "source",
]
df[categorical_cols] = df[categorical_cols].astype("category")
df.dtypes

category        category
amenities            str
bathrooms        float64
bedrooms           Int64
fee             category
has_photo       category
pets_allowed         str
price            float64
square_feet      float64
cityname        category
state           category
latitude         float64
longitude        float64
source          category
dtype: object

In [7]:
df.isnull().sum()

category            0
amenities       16074
bathrooms          65
bedrooms          126
fee                 0
has_photo           0
pets_allowed    60631
price               0
square_feet         2
cityname          302
state             302
latitude           25
longitude          27
source              2
dtype: int64

In [8]:
df["pets_allowed"].value_counts(dropna=False)

pets_allowed
NaN               60631
Cats,Dogs         37212
Cats               1848
Dogs                127
Monthly               2
Cats,Dogs,None        1
Name: count, dtype: int64

In [9]:
# Remove rows with strange values in the "pets_allowed" column
df = df[~df["pets_allowed"].isin(["Monthly", "Cats,Dogs,None"])]
df["pets_allowed"].value_counts(dropna=False)

pets_allowed
NaN          60631
Cats,Dogs    37212
Cats          1848
Dogs           127
Name: count, dtype: int64

In [10]:
pets_lists = (
    df["pets_allowed"]
    .fillna("")
    .str.split(",")
    .apply(lambda x: [item.strip() for item in x if item.strip()])
)

pets_mlb = MultiLabelBinarizer()
pets_encoded = pd.DataFrame(
    pets_mlb.fit_transform(pets_lists),
    columns=[
        f"pet_{name}"
        for name in pets_mlb.classes_
    ],
    index=df.index,
)
pets_encoded

,pet_Cats,pet_Dogs
time,,
2018-12-07 09:20:18,0,0
2018-12-07 09:20:32,0,0
2018-12-07 09:20:44,0,0
2018-12-07 09:20:55,0,0
2018-12-07 09:20:58,0,0
...,...,...
2019-12-26 11:42:47,1,1
2019-12-26 11:42:48,1,1
2019-12-26 12:09:31,0,0


In [11]:
df["amenities"].value_counts(dropna=False)

amenities
NaN                                                                                                           16073
Parking                                                                                                        6204
Parking,Storage                                                                                                2117
Gym,Pool                                                                                                       1875
Pool                                                                                                           1487
                                                                                                              ...  
Dishwasher,Garbage Disposal,Patio/Deck,Washer Dryer,Wood Floors                                                   1
Dishwasher,Garbage Disposal,Patio/Deck,Storage,Washer Dryer                                                       1
Cable or Satellite,Dishwasher,Internet Access,Patio/Deck,Refri

In [12]:
amenities_lists = (
    df["amenities"]
    .fillna("")
    .str.split(",")
    .apply(lambda x: [item.strip() for item in x if item.strip()])
)

amenities_mlb = MultiLabelBinarizer()
amenities_encoded = pd.DataFrame(
    amenities_mlb.fit_transform(amenities_lists),
    columns=[
        f"amenity_{name}"
        for name in amenities_mlb.classes_
    ],
    index=df.index,
)
amenities_encoded

,amenity_AC,amenity_Alarm,amenity_Basketball,amenity_Cable or Satellite,amenity_Clubhouse,amenity_Dishwasher,amenity_Doorman,amenity_Elevator,amenity_Fireplace,amenity_Garbage Disposal,...,amenity_Patio/Deck,amenity_Playground,amenity_Pool,amenity_Refrigerator,amenity_Storage,amenity_TV,amenity_Tennis,amenity_View,amenity_Washer Dryer,amenity_Wood Floors
time,,,,,,,,,,,,,,,,,,,,,
2018-12-07 09:20:18,0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
2018-12-07 09:20:32,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2018-12-07 09:20:44,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2018-12-07 09:20:55,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2018-12-07 09:20:58,0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-26 11:42:47,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2019-12-26 11:42:48,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2019-12-26 12:09:31,1,0,0,1,0,1,0,1,0,0,...,0,0,0,1,0,0,0,0,0,1


In [13]:
df = pd.concat([df.drop(columns=["amenities", "pets_allowed"]), pets_encoded, amenities_encoded], axis=1)
df.head()

,category,bathrooms,bedrooms,fee,has_photo,price,square_feet,cityname,state,latitude,...,amenity_Patio/Deck,amenity_Playground,amenity_Pool,amenity_Refrigerator,amenity_Storage,amenity_TV,amenity_Tennis,amenity_View,amenity_Washer Dryer,amenity_Wood Floors
time,,,,,,,,,,,,,,,,,,,,,
2018-12-07 09:20:18,housing/rent/apartment,1.0,1,No,Thumbnail,1191.0,800.0,Atlanta,GA,33.9407,...,0,0,0,0,0,1,0,0,0,0
2018-12-07 09:20:32,housing/rent/apartment,2.0,4,No,Yes,1450.0,1880.0,Orange Park,FL,30.1728,...,0,0,0,0,0,0,0,0,0,0
2018-12-07 09:20:44,housing/rent/apartment,2.0,3,No,Yes,1395.0,1661.0,Orange Park,FL,30.1728,...,0,0,0,0,0,0,0,0,0,0
2018-12-07 09:20:55,housing/rent/apartment,2.0,3,No,Yes,1255.0,1400.0,Orange Park,FL,30.1728,...,0,0,0,0,0,0,1,0,0,0
2018-12-07 09:20:58,housing/rent/apartment,2.0,2,No,Yes,1693.0,1346.0,Atlanta,GA,33.9407,...,0,0,0,0,0,1,0,0,0,0


In [14]:
df.to_parquet("apartment_rent.parquet")
df = pd.read_parquet("apartment_rent.parquet")
df.head()

,category,bathrooms,bedrooms,fee,has_photo,price,square_feet,cityname,state,latitude,...,amenity_Patio/Deck,amenity_Playground,amenity_Pool,amenity_Refrigerator,amenity_Storage,amenity_TV,amenity_Tennis,amenity_View,amenity_Washer Dryer,amenity_Wood Floors
time,,,,,,,,,,,,,,,,,,,,,
2018-12-07 09:20:18,housing/rent/apartment,1.0,1,No,Thumbnail,1191.0,800.0,Atlanta,GA,33.9407,...,0,0,0,0,0,1,0,0,0,0
2018-12-07 09:20:32,housing/rent/apartment,2.0,4,No,Yes,1450.0,1880.0,Orange Park,FL,30.1728,...,0,0,0,0,0,0,0,0,0,0
2018-12-07 09:20:44,housing/rent/apartment,2.0,3,No,Yes,1395.0,1661.0,Orange Park,FL,30.1728,...,0,0,0,0,0,0,0,0,0,0
2018-12-07 09:20:55,housing/rent/apartment,2.0,3,No,Yes,1255.0,1400.0,Orange Park,FL,30.1728,...,0,0,0,0,0,0,1,0,0,0
2018-12-07 09:20:58,housing/rent/apartment,2.0,2,No,Yes,1693.0,1346.0,Atlanta,GA,33.9407,...,0,0,0,0,0,1,0,0,0,0
